# Bandcamp Artist Discog Scraper
Based on:
- nahnhh's [webcrawl2_movie details.ipynb](https://github.com/nahnhh/top-movies-2005-2025/blob/main/webcrawl2_movie%20details.ipynb)
- diprog's [tls-client-async](https://github.com/diprog/python-tls-client-async) for better client identifiers.
- greasyfork's [Bandcamp: Show more dates](https://greasyfork.org/en/scripts/537005-bandcamp-show-more-dates/code)
- dbeley's [bandcamp-library-scraper.py](https://github.com/dbeley/bandcamp-library-scraper/blob/main/bandcamp-library-scraper.py) for `extract-discography` function

Artist list used: `slushwave-bandcamp-links.txt` compiled in Slushwave Social Club Discord server.

In [1]:
import warnings
warnings.filterwarnings('ignore')
from pathlib import Path
import logging
from rich.logging import RichHandler

FORMAT = '%(message)s'
logging.basicConfig(
    level="INFO", format=FORMAT, datefmt="[%X]", handlers=[RichHandler()]
)
log = logging.getLogger('rich')

### Use `tls-client` instead of plain requests
TLS fingerprinting gives a more browser-like behavior to bypass anti-bot protections.

In [2]:
from bs4 import BeautifulSoup
from async_tls_client import AsyncSession
import asyncio
import os
import random
import json

In [3]:
Path.cwd()

WindowsPath('d:/slushnet/backup')

In [4]:
GOOD_PROFILES = Path(r"D:\slushnet\good_profiles.json")
with open(GOOD_PROFILES, "r") as f:
		OK_CLIENTS = json.load(f)
class BrowserSession:
	def __init__(self):
		self.requests_made = 0
		self.new_session()
		self.retire_after = random.randint(40, 100)

	def rotate_client(self):
		self.client_identifier = random.choice(OK_CLIENTS)

	def new_session(self):
		self.rotate_client()
		self.session = AsyncSession(
			client_identifier=self.client_identifier,
			random_tls_extension_order=True
		)

		self.session.proxies.update({
			"http": os.getenv("mobileproxyuk"),
			"https": os.getenv("mobileproxyuk"),
		})

		self.session.headers.update({
			"Referer": "https://bandcamp.com/",
			"Accept-Language": "en-US,en;q=0.9",
		})

	async def get(self, url, **kwargs):
		if self.requests_made >= self.retire_after:
			self.new_session()
			self.requests_made = 0
			self.retire_after = random.randint(40, 100)

		resp = await self.session.get(url,**kwargs)
		self.requests_made += 1
		return resp

In [5]:
async def fetch(urls):
	random.seed(42)
	s = BrowserSession()
	failed = []
	soups = []
	for url in urls:
		r = await s.get(url)
		soup = BeautifulSoup(r.text, 'lxml') # type: ignore
		if soup.title and soup.title.get_text(strip=True) == "Client Challenge":
			failed.append({
				"url": url,
				"profile": s.client_identifier,
				})
			log.warning(f"Client Challenge with {s.client_identifier} for {url}")
			s.new_session()
			continue
		a = soup.select_one("body a[href]")
		if (a and soup.body and soup.body.get_text().startswith("You are being redirected")):
			redirect_url = a["href"]
			log.info(f"Redirect {url} -> {redirect_url}")
			r = await s.get(redirect_url)
			soup = BeautifulSoup(r.text, 'lxml') # type: ignore
		soups.append(soup)
	return soups

In [6]:
import re
def nozero(text):
	return re.sub(r'[\u200b\u200c\u200d\ufeff]', '', text)

### Fetching soup from urls: music page

In [7]:
urls = [
	'https://illusionarydreaming.bandcamp.com/', # custom grid, no music-grid
	'https://itachitsukiyomi.bandcamp.com/', # 4 columns
	'https://giftsfromhome.bandcamp.com/', # 3 columns
	'https://blackmoon00.bandcamp.com/', # 2 columns
	]

soups = await fetch(urls)

In [8]:
print("Albums fetched:")
print([soup.title.get_text(strip=True) for soup in soups])

Albums fetched:
['Illusionary ドリーミング', 'Music | itachi ツクヨミ', 'Music | Quà từ Nhà', 'Music | blackmoonブラックムーン']


#### Extract album urls from music page

In [9]:
def extract_album_urls(music_page_soup, artist_url=None):
	"""Extract all album urls from the artist's Music page"""
	if not artist_url:
		artist_url = music_page_soup.select_one('meta[property="og:url"]').get('content')
	links = (
		music_page_soup.select("li.music-grid-item a[href]")
		or music_page_soup.select("div.ipCellImage a[href]")
	)
	albums = []
	for a in links:
		href = a["href"]
		if href.startswith("http"):
			albums.append(href)
		else:
			albums.append(artist_url.rstrip("/") + href)

	return albums

In [10]:
extract_album_urls(soups[3])[:3]

['https://blackmoon00.bandcamp.com/album/--4',
 'https://blackmoon00.bandcamp.com/album/--3',
 'https://blackmoon00.bandcamp.com/album/--2']

### Metadata of one track / single

In [11]:
t_url = 'https://eternal984.bandcamp.com/track/reworked'
r = await BrowserSession().get(t_url)
soup = BeautifulSoup(r.text, 'lxml') # type: ignore
t_schema = json.loads(soup.select("script[type='application/ld+json']")[0].get_text(strip=True))
t_schema.keys()

dict_keys(['@type', '@id', 'additionalProperty', 'name', 'description', 'duration', 'dateModified', 'datePublished', 'inAlbum', 'byArtist', 'publisher', 'copyrightNotice', 'keywords', 'image', 'sponsor', 'mainEntityOfPage', '@context'])

In [12]:
[
  t_schema['byArtist']['name'],
	t_schema['name'],
  t_schema['image'],
  t_schema['inAlbum']['numTracks']
]

['e t e r n a l',
 '成立 ( reworked )',
 'https://f4.bcbits.com/img/a0166679623_10.jpg',
 1]

In [13]:
tralbum_tag = soup.select_one("[data-tralbum]")
t_tralbum = json.loads(tralbum_tag.get("data-tralbum", "{}")) # type: ignore
t_tralbum.keys()

dict_keys(['for the curious', 'current', 'preorder_count', 'hasAudio', 'art_id', 'packages', 'defaultPrice', 'freeDownloadPage', 'FREE', 'PAID', 'artist', 'item_type', 'id', 'last_subscription_item', 'has_discounts', 'is_bonus', 'play_cap_data', 'is_purchased', 'items_purchased', 'is_private_stream', 'is_band_member', 'licensed_version_ids', 'package_associated_license_id', 'has_video', 'tralbum_subscriber_only', 'album_is_preorder', 'album_release_date', 'trackinfo', 'playing_from', 'album_url', 'album_upsell_url', 'url'])

In [14]:
t_tralbum.get('art_id')

166679623

In [15]:
t_tralbum['trackinfo'][0].keys()

dict_keys(['id', 'track_id', 'file', 'artist', 'title', 'encodings_id', 'license_type', 'private', 'track_num', 'album_preorder', 'unreleased_track', 'title_link', 'has_lyrics', 'has_info', 'streaming', 'is_downloadable', 'has_free_download', 'free_album_download', 'duration', 'lyrics', 'sizeof_lyrics', 'is_draft', 'video_source_type', 'video_source_id', 'video_mobile_url', 'video_poster_url', 'video_id', 'video_caption', 'video_featured', 'alt_link', 'encoding_error', 'encoding_pending', 'play_count', 'is_capped', 'track_license_id'])

In [16]:
t_tralbum['trackinfo'][0]

{'id': 3497950636,
 'track_id': 3497950636,
 'file': {'mp3-128': 'https://t4.bcbits.com/stream/155ba7a30beb8e9aabae2c4c3d496b57/mp3-128/3497950636?p=0&ts=1782230239&t=e8d2b4494344b521d0552e932f23d52da86b2648&token=1782230239_2934c1bdd2a640179f8d12c0657db80e7d478c21'},
 'artist': None,
 'title': '成立 ( reworked )',
 'encodings_id': 1459088605,
 'license_type': 1,
 'private': None,
 'track_num': None,
 'album_preorder': False,
 'unreleased_track': False,
 'title_link': '/track/reworked',
 'has_lyrics': False,
 'has_info': True,
 'streaming': 1,
 'is_downloadable': True,
 'has_free_download': None,
 'free_album_download': False,
 'duration': 665.798,
 'lyrics': None,
 'sizeof_lyrics': 0,
 'is_draft': False,
 'video_source_type': None,
 'video_source_id': None,
 'video_mobile_url': None,
 'video_poster_url': None,
 'video_id': None,
 'video_caption': None,
 'video_featured': None,
 'alt_link': None,
 'encoding_error': None,
 'encoding_pending': None,
 'play_count': 0,
 'is_capped': False,
 

In [17]:
t_tralbum.get('current').keys()

dict_keys(['audit', 'title', 'new_date', 'mod_date', 'publish_date', 'private', 'killed', 'download_pref', 'require_email', 'is_set_price', 'set_price', 'minimum_price', 'minimum_price_nonzero', 'require_email_0', 'artist', 'about', 'credits', 'auto_repriced', 'new_desc_format', 'band_id', 'selling_band_id', 'art_id', 'download_desc_id', 'track_number', 'release_date', 'file_name', 'lyrics', 'album_id', 'encodings_id', 'pending_encodings_id', 'license_type', 'isrc', 'preorder_download', 'streaming', 'id', 'type'])

In [18]:
t_tralbum['current']['track_number']

### Metadata of one album

#### Fetching soup from urls: album page

In [19]:
urls = [
	'https://memorykeeper7.bandcamp.com/album/t-o-b-e-l-i-e-v-e-i-n-m-y-p-a-i-n',
	# 'https://geometriclullaby.bandcamp.com/album/geo-c07'
	]

soups = await fetch(urls)

print("Albums fetched:")
print([soup.title.get_text(strip=True) for soup in soups])

Albums fetched:
['『t̶o̶ ̶b̶e̶l̶i̶e̶v̶e̶ ̶i̶n̶ ̶m̶y̶ ̶p̶a̶i̶n̶』 | memorykeeper7']


In [20]:
import json
album_schema = json.loads(soups[0].select("script[type='application/ld+json']")[0].get_text(strip=True))
tralbum_tag = soups[0].select_one("[data-tralbum]")
tralbum = json.loads(tralbum_tag.get("data-tralbum", "{}"))

[nozero(album_schema['name']), # album name
 nozero(album_schema['byArtist']['name']), # artist name
 album_schema['numTracks'], # number of tracks
 album_schema['keywords'] # tags
]

['『t̶o̶ ̶b̶e̶l̶i̶e̶v̶e̶ ̶i̶n̶ ̶m̶y̶ ̶p̶a̶i̶n̶』',
 'memorykeeper7',
 5,
 ['Experimental',
  'ambient',
  'ambient vaporwave',
  'ethereal',
  'memories',
  'phaserwave',
  'plunderphonics',
  'slushwave',
  'vaporwave',
  'Norrköping']]

In [21]:
album_schema.keys()

dict_keys(['albumReleaseType', '@id', 'mainEntityOfPage', '@type', 'name', 'dateModified', 'albumRelease', 'byArtist', 'publisher', 'numTracks', 'track', 'image', 'keywords', 'datePublished', 'description', 'creditText', 'copyrightNotice', 'sponsor', 'additionalProperty', '@context'])

In [22]:
album_schema['track']['itemListElement'][0]

{'@type': 'ListItem',
 'position': 1,
 'item': {'@type': 'MusicRecording',
  '@id': 'https://memorykeeper7.bandcamp.com/track/--1124',
  'additionalProperty': [{'@type': 'PropertyValue',
    'name': 'track_id',
    'value': 2338863415},
   {'@type': 'PropertyValue',
    'name': 'license_name',
    'value': 'all_rights_reserved'}],
  'name': '›Ｉ ｎｅｅｄ ｙｏｕ‹',
  'duration': 'P00H18M00S',
  'copyrightNotice': 'All Rights Reserved',
  'mainEntityOfPage': 'https://memorykeeper7.bandcamp.com/track/--1124'}}

In [23]:
album_schema['description']

'𝑀𝑒𝓂𝑜𝓇𝒾𝑒𝓈 𝑜𝒻 𝓂𝓎 𝓅𝒶𝓈𝓉, 𝓂𝒶𝒹𝑒 𝒶𝓃𝑒𝓌.\r\n\r\n⠀⠀⠀⡎⢉⠀⠀⠀⠀⠀⠀⠀⠀⠀\r\n⠀⠀⠀⢈⣁⠆⡀⠀⠀⠀⠀⣄⠀⠀\r\n⠀⠀⠀⢳⢹⠁⠀⠱⡀⠀⠀⢈⠆⠀\r\n⠀⠀⠀⠠⢾⣆⠀⠞⠁⠀⣠⠮⡤⡀\r\n⠀⠀⠀⠀⠐⠹⢦⡀⠀⠰⠁⡀⢰⡁\r\n⠀⠀⠀⢔⠞⠛⠶⣟⣦⣀⠀⠀⠛⠀\r\n⠀⠀⠀⠌⣤⡴⠀⠀⠀⠉⠳⡰⡡⠄\r\n⠀⠀⠀⠀⠀⠀⠀⠀⢀⣀⡀⠹⡄⠀\r\n⠀⠀⠀⠀⠀⠀⠀⠀⡇⢄⠙⢀⢷⢁\r\n⠀⠀⠀⠀⣀⣄⡀⠀⠑⠀⠀⠊⣸⢰\r\n⠀⠀⠀⡔⠁⠠⠗⠀⠀⠀⠀⠀⡭⠄\r\n⠀⠀⠀⢣⠀⠀⠲⣄⣀⣀⢤⡾⠁⠀\r\n⠀⠀⠀⠀⠑⠢⠀⠀⢠⣴⣟⠍⠀⠀\r\n⠐⠄⠄⠤⢐⡢⣀⡼⡾⠋⠀⠀⠀⠀\r\n⠀⠀⢀⠔⣡⣞⠏⠀⠀⠀⠀⠀⠀⠀\r\n⠀⠔⢡⠞⠁⡎⠀⠔⠒⡄⠀⠀⠀⠀\r\n⡎⢰⠃⠀⠀⡗⠄⠁⣀⠆⠀⠀⠀⠀\r\n⠀⠁⢀⠃⠀⠣⡀⠀⠀⠀⠀⣥⠀⠱\r\n⠀⠀⡂⢧⡀⠐⢌⣀⠀⠀⢀⡠⢠⢀\r\n⠀⠀⠂⠀⠁⠀⠀⢐⠩⣏⣋⡸⠗⠊\r\n⠀⠀⠀⠀⠀⠈⠉⡆⢡⠀⠀⠀⠀⠀\r\n⠀⠀⠀⠀⠑⠤⠔⠁⣸⠀⠀⠀⠀⠀\r\n⠀⠀⠀⠀⠀⠀⠀⠀⠥⠃⠀⠀⠀⠀'

In [24]:
album_schema['creditText']

'Self portraits by me, download includes extra photos!\r\n\r\nHeavily inspired by days of blue skies https://daysofblue.bandcamp.com'

#### Get alternative album urls from labels in description/credits

In [25]:
import re

ALT_ALBUM_URL = re.compile(r"https://[a-zA-Z0-9-]+\.bandcamp\.com/album/\S+")

def extract_alt_album_urls(album_schema):
	text = (
		(album_schema.get("creditText") or "") + " " +
		(album_schema.get("description") or "")
	)
	links = ALT_ALBUM_URL.findall(text)
	# dedupe
	seen = set()
	result = []
	for link in links:
		if link not in seen:
			seen.add(link)
			result.append(link)

	return result

In [26]:
extract_alt_album_urls(album_schema)

[]

In [27]:
tralbum.keys()

dict_keys(['for the curious', 'current', 'preorder_count', 'hasAudio', 'art_id', 'packages', 'defaultPrice', 'freeDownloadPage', 'FREE', 'PAID', 'artist', 'item_type', 'id', 'last_subscription_item', 'has_discounts', 'is_bonus', 'play_cap_data', 'is_purchased', 'items_purchased', 'is_private_stream', 'is_band_member', 'licensed_version_ids', 'package_associated_license_id', 'has_video', 'tralbum_subscriber_only', 'featured_track_id', 'initial_track_num', 'is_preorder', 'album_is_preorder', 'album_release_date', 'trackinfo', 'playing_from', 'url', 'use_expando_lyrics'])

In [28]:
tralbum['id']

234642597

In [29]:
tralbum['trackinfo'][0]

{'id': 2338863415,
 'track_id': 2338863415,
 'file': {'mp3-128': 'https://t4.bcbits.com/stream/0da4dad1f20f23346cf031715f638f88/mp3-128/2338863415?p=0&ts=1782230241&t=37fc44ac0a14f353f4a6a65965ac4c5e79046330&token=1782230241_9801ca74522436ca30df9fb5627234873423735a'},
 'artist': None,
 'title': '›Ｉ ｎｅｅｄ ｙｏｕ‹',
 'encodings_id': 233847074,
 'license_type': 1,
 'private': None,
 'track_num': 1,
 'album_preorder': False,
 'unreleased_track': False,
 'title_link': '/track/--1124',
 'has_lyrics': False,
 'has_info': False,
 'streaming': 1,
 'is_downloadable': False,
 'has_free_download': None,
 'free_album_download': False,
 'duration': 1080.12,
 'lyrics': None,
 'sizeof_lyrics': 0,
 'is_draft': False,
 'video_source_type': None,
 'video_source_id': None,
 'video_mobile_url': None,
 'video_poster_url': None,
 'video_id': None,
 'video_caption': None,
 'video_featured': None,
 'alt_link': None,
 'encoding_error': None,
 'encoding_pending': None,
 'play_count': None,
 'is_capped': None,
 'trac

In [30]:
tralbum['trackinfo'][0]['title_link']

'/track/--1124'

In [31]:
# Get all track urls
import isodate
import pandas as pd
df = pd.DataFrame([
	{
		"url": t['item']['@id']
	}
	for t in album_schema["track"]["itemListElement"]
])
print(df)

                                                 url
0    https://memorykeeper7.bandcamp.com/track/--1124
1    https://memorykeeper7.bandcamp.com/track/--1129
2  https://memorykeeper7.bandcamp.com/track/s-t-a...
3       https://memorykeeper7.bandcamp.com/track/c-r
4    https://memorykeeper7.bandcamp.com/track/--1130


#### Get unique track arts
This is how bandcamp art ID works:
1. Bandcamp generates a unique art_ID for each time you upload a track art.
2. The rest uses art_ID from the album art.

We can drop duplicates based on image hash, with some cases we have to delete manually (dobs album).

In [32]:
import re
import hashlib

async def get_art_id(url):
	session = BrowserSession()
	r = await session.get(url)
	soup = BeautifulSoup(r.text or "", "lxml")

	icon_url = soup.select_one('link[rel="shortcut icon"]')['href'] # type: ignore

	img = await session.get(icon_url)
	img_hash = hashlib.blake2b(img.content, digest_size=8).hexdigest() # type: ignore

	art_id = re.search(r'a(\d+)_', icon_url).group(1) # type: ignore

	return art_id, img_hash


results = await asyncio.gather(
    *(get_art_id(url) for url in df["url"])
)

art_df = pd.DataFrame(
    results,
    columns=["art_id", "img_hash"]
)

print(art_df)

       art_id          img_hash
0  3715886967  cfd4c303d4b0ef0f
1  0723863566  42dadba7149763e9
2  0979312607  68e5831d7d968a61
3  0923106527  a6c2cc92d4590627
4  0329421763  a022ae0da09037bf


In [33]:
print(f"Total unique art IDs: {art_df['art_id'].nunique()}, Unique Image Hashes: {art_df['img_hash'].nunique()}")

Total unique art IDs: 5, Unique Image Hashes: 5


#### Get all dates of the album

In [34]:
tralbum = json.loads(tralbum_tag["data-tralbum"])['current']
tralbum2 = json.loads(tralbum_tag["data-tralbum"])['trackinfo']

In [35]:
[nozero(tralbum['title']), # album name
 tralbum['art_id'], # album image id: 2569838638 in https://f4.bcbits.com/img/a2569838638_10.jpg
 tralbum['new_date'], # date first created draft
 tralbum['mod_date'], # date last modified
 tralbum['publish_date'], # publication date
 tralbum['release_date'] # release date
 ]

['『t̶o̶ ̶b̶e̶l̶i̶e̶v̶e̶ ̶i̶n̶ ̶m̶y̶ ̶p̶a̶i̶n̶』',
 296383690,
 '18 Mar 2026 02:30:37 GMT',
 '20 Mar 2026 13:50:18 GMT',
 '20 Mar 2026 13:50:18 GMT',
 '20 Mar 2026 13:50:18 GMT']

In [36]:
# Get all track durations
import pandas as pd
track_df = pd.DataFrame([
	{
		"position": t['track_num'],
		"duration": t['duration'],
	}
	for t in tralbum2
])
print(track_df)

   position  duration
0         1  1080.120
1         2   900.125
2         3  1020.120
3         4   840.125
4         5   900.125


#### Adding up total runtime of the album:

In [37]:
from datetime import timedelta
print(timedelta(seconds=int(track_df["duration"].sum())))

1:19:00
